# Bài 21: Giao diện web

## Từ CLI đến ứng dụng web

Bài 19 đã theo dõi cách mọi thứ kết nối. Bài 20 đã dạy bạn kiến thức web cơ bản — frontend/backend, API, JSON, SSE. Giờ là **phía người dùng** — giao diện web thực tế.

Giao diện web là **cách chính để sử dụng sản phẩm này**. Mở `http://localhost:5173` trong trình duyệt:

- Chat với team trong giao diện web gọn gàng
- Xem phản hồi streaming (hiển thị theo thời gian thực) — SSE mà bạn đã học ở Bài 20
- Duyệt và đọc các bài viết đã tạo từ thanh bên (sidebar)
- Xóa các bài viết không cần

Kiến trúc: **FastAPI backend** (AgentOS) + **React frontend** (Vite).

## AgentOS — Backend trong 30 dòng

Agno cung cấp **AgentOS** — một wrapper biến bất kỳ Agent hoặc Team nào thành một FastAPI server với hơn 50 endpoint.

```python
from agno.os import AgentOS
from agents.team import team

agent_os = AgentOS(teams=[team])
app = agent_os.get_app()
```

Chỉ vậy thôi. AgentOS tự động tạo:
- `POST /teams/seo-workspace/runs` — gửi prompt (hỗ trợ SSE streaming)
- `GET /health` — kiểm tra sức khỏe server
- `GET /docs` — tài liệu API tương tác (Swagger)
- Quản lý session, lịch sử chạy, và nhiều hơn nữa

File `serve.py` của chúng ta thêm các route tùy chỉnh cho tầng lưu trữ bài viết (article storage) lên trên đó.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Let's read the actual serve.py
serve_path = os.path.abspath("../../output/backend/serve.py")
with open(serve_path, "r", encoding="utf-8") as f:
    print(f.read())

## Phân tích serve.py

File gồm 4 phần:

1. **Xác thực API key** — thoát nếu thiếu ANTHROPIC_API_KEY
2. **FastAPI app tùy chỉnh** — với CORS middleware (cho phép React frontend kết nối) và các route cho bài viết
3. **AgentOS wrapper** — `AgentOS(teams=[team], base_app=app)` gộp các route tùy chỉnh với các route tự sinh
4. **Khởi chạy server** — `agent_os.serve(app="serve:app", port=7777, reload=True)`

Các route bài viết tùy chỉnh là wrapper mỏng quanh các hàm trong `tools.storage`:
- `GET /api/articles` → gọi `list_articles()`
- `GET /api/articles/{id}` → gọi `get_article()`
- `DELETE /api/articles/{id}` → xóa khỏi metadata + xóa file `.md`

## Frontend

React frontend nằm trong `output/frontend/`:

```
output/frontend/
├── package.json          Dependencies (react, react-markdown, vite)
├── vite.config.js        Dev server proxy → localhost:7777
├── index.html
└── src/
    ├── main.jsx          Điểm vào ứng dụng
    ├── App.jsx           Layout: sidebar + vùng chính
    ├── api.js            API client (SSE streaming + các hàm fetch)
    └── components/
        ├── Chat.jsx      Giao diện chat với SSE streaming + thẻ bài viết
        ├── ArticleList.jsx  Danh sách bài viết trong sidebar (badge NEW)
        ├── ArticleView.jsx  Trang xem bài viết đầy đủ
        └── Toast.jsx     Thông báo toast
```

**SSE (Server-Sent Events)** là cách frontend nhận phản hồi streaming — bạn đã học khái niệm này ở Bài 20. `Chat.jsx` dùng endpoint gốc của AgentOS tại `/teams/seo-workspace/runs` với `stream=true` để hiển thị phản hồi theo thời gian thực.

**Vite proxy** (cũng từ Bài 20): `vite.config.js` proxy các request `/api` và `/teams` đến `localhost:7777` (backend), nên frontend chỉ cần dùng URL tương đối.

## Chạy ứng dụng web

Cách dễ nhất — một lệnh duy nhất:

```bash
python output/start.py
```

Khởi chạy cả backend (port 7777) và frontend (port 5173). Mở `http://localhost:5173` trong trình duyệt. Ctrl+C dừng cả hai.

Hoặc chạy riêng:

**Terminal 1 — Backend:**
```bash
python output/backend/serve.py
```
Tài liệu Swagger tại `http://localhost:7777/docs`.

**Terminal 2 — Frontend:**
```bash
cd output/frontend
npm install    # chỉ lần đầu
npm run dev
```
Mở `http://localhost:5173` trong trình duyệt.

In [ ]:
print("Để chạy ứng dụng web:\n")
print("  Một lệnh:    python output/start.py")
print()
print("  Hoặc chạy riêng:")
print("    Backend:   python output/backend/serve.py")
print("    Frontend:  cd output/frontend && npm run dev")
print()
print("Backend: http://localhost:7777")
print("  /docs      -- Tài liệu Swagger API")
print("  /health    -- Kiểm tra sức khỏe server")
print("  /api/articles -- Danh sách bài viết")
print()
print("Frontend: http://localhost:5173")
print("  Chat với team, duyệt bài viết")

## Vibecoding backend

Đây là cách `serve.py` được tạo ra — **không phải viết tay, mà bằng cách mô tả cho Claude Code**.

Quy trình:
1. File `CLAUDE.md` đã mô tả toàn bộ kiến trúc (agents, tools, storage)
2. Chúng ta bảo Claude Code: *"Tạo một FastAPI backend dùng AgentOS để phục vụ team, với các route tùy chỉnh cho tầng lưu trữ bài viết"*
3. Claude Code đọc code hiện có, hiểu các pattern, và tạo ra `serve.py`

Điểm mấu chốt: **bạn không cần biết cú pháp FastAPI hay AgentOS**. Bạn mô tả những gì bạn muốn dựa trên code đã có, và Claude Code tạo ra phần triển khai.

Lý do nó hoạt động tốt:
- **CLAUDE.md** cho Claude Code đầy đủ ngữ cảnh về dự án
- **Code hiện có** (team.py, storage.py) sạch và có tổ chức
- **Prompt** cụ thể: *team nào, route nào, port nào*

In [ ]:
# Let's see the CLAUDE.md that gives Claude Code context
claude_md_path = os.path.abspath("../../CLAUDE.md")
with open(claude_md_path, "r", encoding="utf-8") as f:
    content = f.read()

print(f"CLAUDE.md ({len(content.splitlines())} dòng)\n")
print(content[:3000])
if len(content) > 3000:
    print("\n... (lược bớt)")

## Ghi chú: json.loads() và json.dumps()

Bạn sẽ thấy hai hàm này khi làm việc với dữ liệu JSON trong Python:

- `json.loads(text)` — **load string**: chuyển chuỗi JSON thành dict/list Python
- `json.dumps(data)` — **dump string**: chuyển dict/list Python thành chuỗi JSON

```python
import json

# Chuỗi JSON -> dict Python
text = '{"title": "SEO Guide", "words": 1500}'
data = json.loads(text)
print(data["title"])  # "SEO Guide"

# Dict Python -> chuỗi JSON
back_to_text = json.dumps(data)
print(back_to_text)   # '{"title": "SEO Guide", "words": 1500}'
```

Chữ 's' trong loads/dumps là viết tắt của 'string'. Ngoài ra còn có `json.load(file)` / `json.dump(data, file)` để đọc/ghi file trực tiếp.

## Đọc code sản phẩm

### Bản đồ: Bài học → File sản phẩm

| Bạn đã học | Bài học | File sản phẩm |
|---|---|---|
| Cách LLM hoạt động, prompt, model | 05-07 | (Kiến thức nền — định hướng mọi quyết định) |
| Tạo agent, tools | 08-09 | `output/backend/agents/content_writer.py`, `image_finder.py`, `aio_analyzer.py` |
| Structured output (Pydantic) | 10-11 | (Dùng trong phân tích AIO) |
| Lưu trữ file cục bộ | 12 | `output/backend/tools/storage.py` |
| Claude Code, prompting, kế hoạch | 13-15 | (Kỹ năng — cách bạn làm việc với Claude Code) |
| Content Writer, Images, AIO | 16-17 | `output/backend/agents/content_writer.py`, `image_finder.py`, `aio_analyzer.py` |
| Team và xử lý hàng loạt | 18 | `output/backend/agents/team.py` |
| Cách mọi thứ kết nối | 19 | Tất cả file — toàn bộ luồng yêu cầu |
| Kiến thức web cơ bản | 20 | (Kiến thức nền — frontend, backend, API, SSE) |
| Giao diện web (bài này) | 21 | `output/backend/serve.py` + `output/frontend/` |

### Cấu trúc file

```
output/
├── start.py                        Khởi chạy backend + frontend cùng lúc
├── backend/                        Python backend
│   ├── serve.py                    Backend web (AgentOS + route bài viết)
│   ├── agents/                     Định nghĩa agent (ai làm gì)
│   │   ├── __init__.py             Re-exports
│   │   ├── content_writer.py       Content Writer
│   │   ├── image_finder.py         Image Finder
│   │   ├── aio_analyzer.py         AIO Analyzer
│   │   └── team.py                 Agno Team
│   └── tools/                      Định nghĩa tool (làm gì được)
│       ├── __init__.py             Package marker
│       ├── storage.py              Lưu trữ file cục bộ
│       ├── search.py               DataForSEO web search
│       ├── aio.py                  Phân tích AIO + credentials
│       └── images.py               DataForSEO image search
└── frontend/                       React + Vite web app
    ├── package.json                Dependencies
    ├── vite.config.js              Proxy → backend
    └── src/                        Components + API client
        └── components/
            ├── Chat.jsx            SSE streaming + thẻ bài viết
            ├── ArticleList.jsx     Sidebar (badge NEW)
            ├── ArticleView.jsx     Trình đọc bài viết
            └── Toast.jsx           Thông báo
```

## Tổng kết

Giao diện web là **giao diện chính** của sản phẩm:

- **`serve.py`** bọc team bằng AgentOS (50+ endpoint tự sinh) + route bài viết tùy chỉnh
- **React frontend** trong `output/frontend/` — chat với SSE streaming, sidebar bài viết, trình đọc Markdown
- **`python output/start.py`** khởi chạy cả backend và frontend trong một lệnh
- Vite proxy kết nối frontend (port 5173) đến backend (port 7777)

**Bài tiếp theo:** Mở rộng sản phẩm — thêm khả năng mới bằng vibecoding.

## Bài tập

1. Đọc `output/backend/serve.py` và xác định: có bao nhiêu route tùy chỉnh? Mỗi route làm gì?
2. File `api.js` của frontend xử lý SSE streaming. Nó gọi endpoint nào để chat? (Gợi ý: tìm hàm `streamTeamRun`)
3. Nếu bạn muốn thêm route `PUT /api/articles/{id}` để cập nhật metadata bài viết, hàm storage nào sẽ được gọi?
4. Giải thích tại sao Vite proxy cần thiết. Nếu không có proxy, điều gì sẽ xảy ra khi frontend gọi `/api/articles`?

<details>
<summary>Bấm để xem đáp án</summary>

1. **3 route tùy chỉnh:** `GET /api/articles` (liệt kê tất cả), `GET /api/articles/{id}` (lấy một bài), `DELETE /api/articles/{id}` (xóa một bài). Chat dùng endpoint gốc của AgentOS tại `/teams/seo-workspace/runs`.

2. **Endpoint chat:** `POST /teams/seo-workspace/runs` với `stream=true` dưới dạng form-encoded parameter. Frontend đọc các SSE event từ response bằng `fetch()` + `ReadableStream`.

3. **Route PUT:** Nó sẽ gọi `update_article_content(article_id, article_markdown)` từ `tools.storage`. Bạn cũng cần cập nhật các trường metadata trong `articles.json` trực tiếp hoặc thêm một hàm storage mới.

4. **Vite proxy:** Frontend chạy ở `localhost:5173`, backend ở `localhost:7777`. Không có proxy, trình duyệt sẽ gửi request `/api/articles` đến `localhost:5173` (frontend server), không tìm thấy route, và trả về lỗi 404. Proxy chuyển tiếp các request `/api` và `/teams` đến backend, giúp frontend dùng URL tương đối mà không cần biết backend ở port nào.
</details>